In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import zipfile

In [2]:
path = '/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/homer_bart_2.zip'
zip_object = zipfile.ZipFile(file = path, mode = 'r')
zip_object.extractall('./')
zip_object.close()

In [3]:
BATCH_SIZE = 32
IMG_SIZE = (180, 180)

In [4]:
train_dir = '/content/homer_bart_2/training_set'
validation_dir = '/content/homer_bart_2/test_set'

In [5]:
train_dataset = tf.keras.utils.image_dataset_from_directory(train_dir,
                                                            shuffle=True,
                                                            batch_size=BATCH_SIZE,
                                                            image_size=IMG_SIZE)

Found 215 files belonging to 2 classes.


In [6]:
test_dataset = tf.keras.utils.image_dataset_from_directory(validation_dir,
                                                                 shuffle=True,
                                                                 batch_size=BATCH_SIZE,
                                                                 image_size=IMG_SIZE)

Found 54 files belonging to 2 classes.


In [7]:
class_names = train_dataset.class_names
print(class_names)

['bart', 'homer']


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

In [10]:
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.2),
])

In [ ]:
for image, _ in train_dataset.take(1):
  plt.figure(figsize=(10, 10))
  first_image = image[0]
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
    plt.imshow(augmented_image[0] / 255)
    plt.axis('off')

In [12]:
rescale = tf.keras.layers.Rescaling(1./127.5, offset=-1)

In [23]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

In [13]:
IMG_SHAPE = IMG_SIZE + (3,)
base_model = tf.keras.applications.ResNet50(input_shape=IMG_SHAPE,
                                               include_top=False,
                                               weights='imagenet')

In [14]:
image_batch, label_batch = next(iter(train_dataset))
feature_batch = base_model(image_batch)
print(feature_batch.shape)

(32, 6, 6, 2048)


In [15]:
base_model.trainable = False

In [ ]:
base_model.summary()

In [17]:
base_model.output

<KerasTensor: shape=(None, 6, 6, 2048) dtype=float32 (created by layer 'conv5_block3_out')>

In [18]:
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
feature_batch_average = global_average_layer(feature_batch)
print(feature_batch_average.shape)

(32, 2048)


In [19]:
prediction_layer = tf.keras.layers.Dense(1)
prediction_batch = prediction_layer(feature_batch_average)
print(prediction_batch.shape)

(32, 1)


In [24]:
inputs = tf.keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = global_average_layer(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = prediction_layer(x)
model = tf.keras.Model(inputs, outputs)

In [25]:
base_learning_rate = 0.0001
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [26]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 180, 180, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 180, 180, 3)       0         
                                                                 
 tf.math.truediv (TFOpLambd  (None, 180, 180, 3)       0         
 a)                                                              
                                                                 
 tf.math.subtract (TFOpLamb  (None, 180, 180, 3)       0         
 da)                                                             
                                                                 
 resnet50 (Functional)       (None, 6, 6, 2048)        23587712  
                                                                 
 global_average_pooling2d (  (None, 2048)              0     

In [27]:
initial_epochs = 50

loss0, accuracy0 = model.evaluate(test_dataset)

2/2 [==============================] - 4s 756ms/step - loss: 0.7101 - accuracy: 0.5185


In [28]:
print("initial loss: {:.2f}".format(loss0))
print("initial accuracy: {:.2f}".format(accuracy0))

initial loss: 0.71
initial accuracy: 0.52


In [29]:
history = model.fit(train_dataset,
                    epochs=initial_epochs,
                    validation_data=test_dataset)

Epoch 1/50
7/7 [==============================] - 7s 224ms/step - loss: 0.6969 - accuracy: 0.6186 - val_loss: 0.7125 - val_accuracy: 0.5185
Epoch 2/50
7/7 [==============================] - 1s 108ms/step - loss: 0.7063 - accuracy: 0.6140 - val_loss: 0.7274 - val_accuracy: 0.5185
Epoch 3/50
7/7 [==============================] - 1s 107ms/step - loss: 0.7128 - accuracy: 0.6279 - val_loss: 0.7371 - val_accuracy: 0.5185
Epoch 4/50
7/7 [==============================] - 1s 106ms/step - loss: 0.7040 - accuracy: 0.6093 - val_loss: 0.7408 - val_accuracy: 0.5185
Epoch 5/50
7/7 [==============================] - 1s 106ms/step - loss: 0.7195 - accuracy: 0.6093 - val_loss: 0.7459 - val_accuracy: 0.5185
Epoch 6/50
7/7 [==============================] - 1s 107ms/step - loss: 0.7151 - accuracy: 0.6047 - val_loss: 0.7379 - val_accuracy: 0.5185
Epoch 7/50
7/7 [==============================] - 1s 134ms/step - loss: 0.6999 - accuracy: 0.6233 - val_loss: 0.7308 - val_accuracy: 0.5185
Epoch 8/50
7/7 [====

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0,1.0])
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [31]:
base_model.trainable = True

In [37]:
print("Número de camadas no modelo base: ", len(base_model.layers))

fine_tune_at = 120

for layer in base_model.layers[:fine_tune_at]:
  layer.trainable = False

Número de camadas no modelo base:  175


In [38]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer = tf.keras.optimizers.RMSprop(learning_rate=base_learning_rate/10),
              metrics=['accuracy'])

In [ ]:
fine_tune_epochs = 100
total_epochs =  initial_epochs + fine_tune_epochs

history_fine = model.fit(train_dataset,
                         epochs=total_epochs,
                         initial_epoch=history.epoch[-1],
                         validation_data=test_dataset)

In [44]:
acc += history_fine.history['accuracy']
val_acc += history_fine.history['val_accuracy']

loss += history_fine.history['loss']
val_loss += history_fine.history['val_loss']

In [ ]:
plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.ylim([0.8, 1])
plt.plot([initial_epochs-1,initial_epochs-1],
          plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.ylim([0, 1.0])
plt.plot([initial_epochs-1,initial_epochs-1],
         plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [47]:
loss, accuracy = model.evaluate(test_dataset)
print('Teste accuracy :', accuracy)

2/2 [==============================] - 0s 79ms/step - loss: 0.5720 - accuracy: 0.8704
Teste accuracy : 0.8703703880310059


In [40]:
modelo = model.to_json()
with open('modelo.json', 'w') as json_file:
  json_file.write(modelo)

In [41]:
from keras.models import save_model
model_save = save_model(model, 'weights.hdf5')

<ipython-input-41-1310051c93f8>:2: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  model_save = save_model(model, 'weights.hdf5')


In [ ]:
with open('modelo.json') as json_file:
  json_saved_model = json_file.read()
json_saved_model

In [43]:
model_loaded = tf.keras.models.model_from_json(json_saved_model)
model_loaded.load_weights('/content/weights.hdf5')
model_loaded.compile(loss = 'binary_crossentropy',
                     optimizer = 'adam',
                     metrics = ['accuracy'])